In [3]:
import json
import torchaudio as ta
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from rich.progress import track

In [4]:
ds_dir = Path('/media/Data/jp_dir/sounds/vctk_words_v2')

with open(ds_dir / 'dataset_metadata.json') as f:
    metadata = json.load(f)
print(f"got {len(metadata['samples'])} samples in {str(ds_dir)}")

samples = metadata['samples']
labels = []
for s in samples:
    with open(ds_dir / s['label_file']) as f:
        label_dict = json.load(f)
    labels.append(label_dict)

labels_df = pd.DataFrame(labels)
labels_df

got 250620 samples in /media/Data/jp_dir/sounds/vctk_words_v2


,word,start_time,end_time,duration,speaker_id,original_text_id,original_text,word_index,sample_id
0,Please,0.24,0.80,0.56,p225,p225_001,Please call Stella.,0,p225_p225_001_word000
1,call,0.80,1.12,0.32,p225,p225_001,Please call Stella.,1,p225_p225_001_word001
2,Stella,1.12,1.60,0.48,p225,p225_001,Please call Stella.,2,p225_p225_001_word002
3,Ask,0.96,1.28,0.32,p225,p225_002,Ask her to bring these things with her from th...,0,p225_p225_002_word000
4,her,1.28,1.52,0.24,p225,p225_002,Ask her to bring these things with her from th...,1,p225_p225_002_word001
...,...,...,...,...,...,...,...,...,...
250615,one,1.36,1.60,0.24,s5,s5_400,I saw one man vote five times.,2,s5_s5_400_word002
250616,man,1.60,1.92,0.32,s5,s5_400,I saw one man vote five times.,3,s5_s5_400_word003
250617,vote,1.92,2.24,0.32,s5,s5_400,I saw one man vote five times.,4,s5_s5_400_word004
250618,five,2.32,2.64,0.32,s5,s5_400,I saw one man vote five times.,5,s5_s5_400_word005


In [9]:
speaker_info_csv = Path("/media/Data/jp_dir/sounds/vctk_original/speaker-info.csv")
speaker_df = pd.read_csv(speaker_info_csv).drop(columns=["COMMENTS"])
speaker_df

,ID,AGE,GENDER,ACCENTS,REGION
0,p225,23,F,English,Southern England
1,p226,22,M,English,Surrey
2,p227,38,M,English,Cumbria
3,p228,22,F,English,Southern England
4,p229,23,F,English,Southern England
...,...,...,...,...,...
105,p363,22,M,Canadian,Toronto
106,p364,23,M,Irish,Donegal
107,p374,28,M,Australian English,NaN
108,p376,22,M,Indian,NaN


In [10]:
full_labels = labels_df.merge(
    speaker_df,
    how="left",
    left_on="speaker_id",
    right_on="ID",
    validate="m:1",
)

full_labels

,word,start_time,end_time,duration,speaker_id,original_text_id,original_text,word_index,sample_id,ID,AGE,GENDER,ACCENTS,REGION
0,Please,0.24,0.80,0.56,p225,p225_001,Please call Stella.,0,p225_p225_001_word000,p225,23,F,English,Southern England
1,call,0.80,1.12,0.32,p225,p225_001,Please call Stella.,1,p225_p225_001_word001,p225,23,F,English,Southern England
2,Stella,1.12,1.60,0.48,p225,p225_001,Please call Stella.,2,p225_p225_001_word002,p225,23,F,English,Southern England
3,Ask,0.96,1.28,0.32,p225,p225_002,Ask her to bring these things with her from th...,0,p225_p225_002_word000,p225,23,F,English,Southern England
4,her,1.28,1.52,0.24,p225,p225_002,Ask her to bring these things with her from th...,1,p225_p225_002_word001,p225,23,F,English,Southern England
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250615,one,1.36,1.60,0.24,s5,s5_400,I saw one man vote five times.,2,s5_s5_400_word002,s5,22,F,British,NaN
250616,man,1.60,1.92,0.32,s5,s5_400,I saw one man vote five times.,3,s5_s5_400_word003,s5,22,F,British,NaN
250617,vote,1.92,2.24,0.32,s5,s5_400,I saw one man vote five times.,4,s5_s5_400_word004,s5,22,F,British,NaN
250618,five,2.32,2.64,0.32,s5,s5_400,I saw one man vote five times.,5,s5_s5_400_word005,s5,22,F,British,NaN


In [11]:
labels_out_path = ds_dir / "labels.csv"
full_labels.to_csv(labels_out_path, index=False)